# Modeling: predicting repeat orders within 30 days

Goal: rank customers by how likely they are to order again in the next 30 days, so the sales team can follow up with the most promising ones first.

Approach
1. Split the data by time (train, validation, test).
2. Compare three methods: a no-model rule, logistic regression, and LightGBM.
3. Choose the model on the validation set, then report the final result once on the test set.

In [3]:
import sys
sys.path.append("..")

import joblib
import pandas as pd
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from src.evaluate import report

df = pd.read_parquet("../data/processed/dataset.parquet")
cuts = sorted(df["cutoff"].unique())

train = df[df["cutoff"].isin(cuts[:13])]     #Jun 2010 to Jun 2011
valid = df[df["cutoff"].isin(cuts[13:15])]   #Jul and Augs 2011
test = df[df["cutoff"].isin(cuts[15:])]      #Sep to Nov 2011

DROP = ["customer_id", "target", "cutoff", "cutoff_month"]
feats = [c for c in df.columns if c not in DROP]

for name, d in [("train", train), ("valid", valid), ("test", test)]:
    print(name, d.shape, "base rate:", round(d["target"].mean(), 3))
print(len(feats), "fitur")

train (51370, 22) base rate: 0.213
valid (10133, 22) base rate: 0.161
test (16269, 22) base rate: 0.224
18 fitur


## Time-based split

The model learns from the past and is tested on later periods, which mirrors how it will be used in practice. A random split would let the model see customers from the same months in both train and test.

| Set | Cutoffs | Rows | Base rate |
|---|---|---|---|
| Train | Jun 2010 to Jun 2011 (13 cutoffs) | 51,370 | 21.3% |
| Validation | Jul and Aug 2011 (2 cutoffs) | 10,133 | 16.1% |
| Test | Sep to Nov 2011 (3 cutoffs) | 16,269 | 22.4% |

Columns not used as features: customer_id (identifier), target (the answer), cutoff (a date label) and cutoff_month. The last one has the same value for every customer on a given cutoff date, so it cannot help rank customers within that date. A test without it gave the same result, so it was removed. This leaves 18 features.

The base rate differs across periods (16.1% to 22.4%), so lift (which divides by the base rate) is used to compare results fairly.

The validation set is used to make decisions (model choice, when to stop training). The test set is used only once at the end.

## Baselines

Two simple reference points that the final model must beat:

1. Recency rule (no model): customers who bought more recently get a higher score.
2. Logistic regression: a simple model that gives each feature a fixed weight. Features are standardized first, missing values are filled with 0, and class_weight="balanced" is used because only about 21% of rows are repeat customers.

In [4]:
# pembanding 1 : aturan tanpa model (makin baru belanja, makin tinggi skor)
report("Aturan recency", valid["target"], -valid["recency"])

# pembanding 2 : logistic regression
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
lr.fit(train[feats].fillna(0), train["target"])
report("LogisticRegression", valid["target"], lr.predict_proba(valid[feats].fillna(0))[:, 1])

Aturan recency
  PR-AUC : 0.355 (base rate 0.161 )
 ROC=AUC 0.776
 top 10% (precision, recall, lift): (0.401, 0.249, 2.49)
 top 20% (precision, recall, lift): (0.376, 0.466, 2.33)
LogisticRegression
  PR-AUC : 0.548 (base rate 0.161 )
 ROC=AUC 0.821
 top 10% (precision, recall, lift): (0.62, 0.385, 3.85)
 top 20% (precision, recall, lift): (0.46, 0.571, 2.85)


## Baseline results (validation, top 20% of customers)

| Method | PR-AUC | ROC-AUC | Precision | Recall | Lift |
|---|---|---|---|---|---|
| Recency rule | 0.355 | 0.776 | 0.376 | 0.466 | 2.33 |
| Logistic regression | 0.548 | 0.821 | 0.460 | 0.571 | 2.85 |

Validation base rate: 16.1%.

Reading the table
- Even the simple rule without a model beats random selection (lift 2.33).
- Logistic regression adds a large improvement over the rule (lift 2.85), so a model clearly adds value.
- These two results are the bar that LightGBM has to beat.

## LightGBM

LightGBM builds many small decision trees in sequence, where each new tree corrects the errors of the previous ones. Key settings:
- n_estimators=600 (maximum number of trees) with learning_rate=0.03 (small, careful steps)
- num_leaves=31 (maximum complexity of each tree)
- class_weight="balanced" (give the rarer repeat class more weight)
- Early stopping: training stops when validation AUC has not improved for 50 trees, and the best number of trees is kept.

In [5]:
model = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.03, num_leaves=31,
                           class_weight="balanced", random_state=42, verbose=-1)
model.fit(train[feats], train["target"],
          eval_set=[(valid[feats], valid["target"])], eval_metric="auc",
          callbacks=[lgb.early_stopping(50, verbose=False)])

print("iterasi terbaik:", model.best_iteration_)
report("LightGBM (validasi)", valid["target"], model.predict_proba(valid[feats])[:, 1])

c:\Users\APTIKA\OneDrive\Documents\RIMA\customer_repeat_order\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


iterasi terbaik: 29
LightGBM (validasi)
  PR-AUC : 0.559 (base rate 0.161 )
 ROC=AUC 0.828
 top 10% (precision, recall, lift): (0.622, 0.386, 3.86)
 top 20% (precision, recall, lift): (0.475, 0.589, 2.95)


## Validation results: three methods compared (top 20% of customers)

| Method | PR-AUC | ROC-AUC | Precision | Recall | Lift |
|---|---|---|---|---|---|
| Recency rule | 0.355 | 0.776 | 0.376 | 0.466 | 2.33 |
| Logistic regression | 0.548 | 0.821 | 0.460 | 0.571 | 2.85 |
| LightGBM | 0.559 | 0.828 | 0.475 | 0.589 | 2.95 |

Validation base rate: 16.1%. Best iteration of LightGBM: 29 trees.

Reading the table
- The biggest jump is from the simple rule to logistic regression (lift 2.33 to 2.85). The model clearly adds value.
- LightGBM is only slightly better than logistic regression (lift 2.95 vs 2.85). With about 2,000 customers in the top 20%, precision has a margin of error of about 2 points, so a difference of 0.015 is within noise.
- Only 29 trees were needed, which suggests the patterns in this data are fairly simple. Better features would likely help more than a more complex algorithm.

## Final test (run once)

The test set (Sep to Nov 2011) has not been used for any decision so far. It is evaluated once, after the model and features are fixed. Running it repeatedly while changing the model would turn it into a second validation set and make the result over-optimistic.

In [6]:
report("LightGBM (test)", test["target"], model.predict_proba(test[feats])[:, 1])

LightGBM (test)
  PR-AUC : 0.569 (base rate 0.224 )
 ROC=AUC 0.789
 top 10% (precision, recall, lift): (0.683, 0.304, 3.05)
 top 20% (precision, recall, lift): (0.55, 0.491, 2.45)


## Test results (top 20% of customers)

| Method | PR-AUC | ROC-AUC | Precision | Recall | Lift |
|---|---|---|---|---|---|
| LightGBM | 0.569 | 0.789 | 0.550 | 0.491 | 2.45 |

Test base rate: 22.4%.

Contacting the top 20% of customers reaches about 49% of all customers who will reorder, and the customers chosen are about 2.45 times more likely to reorder than customers chosen at random.

Note: the validation lift (2.95) is higher than the test lift (2.45). The periods have different base rates, and the validation score is slightly optimistic because it was also used to decide when to stop training. The test number is the one to report.

In [7]:
joblib.dump({"model": model, "features": feats}, "../data/processed/model.pkl")

['../data/processed/model.pkl']

## Summary

- Main model: LightGBM, chosen for its slightly higher validation scores, native handling of missing values, and compatibility with SHAP. Logistic regression remains a simple alternative with almost the same performance.
- Result on unseen future data (test): lift 2.45 and recall 49% in the top 20% of customers.
- cutoff_month was removed because it cannot rank customers within a single cutoff date and did not improve results.
- Limitation: the model uses class_weight="balanced", so its scores are good for ranking customers but should not be read as exact probabilities.

Next: evaluation by decile and by cutoff date, SHAP explanations, and a business impact simulation.